In [ ]:
%load_ext autoreload
%autoreload 2

: 

In [22]:
import os
os.chdir('/home/stud/ath/ath_ws/keypoint_matcher')

from config import config
from utils import logger
from utils import show_batch

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# torch.cuda.empty_cache()

# Check Data Module

In [ ]:
from src import MatchesDataModule
dm = MatchesDataModule()

dm.setup(stage='fit')

In [24]:
dl = dm.train_dataloader()

In [25]:
batch = next(iter(dl))

In [ ]:
reference_patches = batch.reference_patches
target_patches = batch.target_patches
print(reference_patches.shape, target_patches.shape) 

patch_level_reference_coords = batch.patch_level_reference_coords
patch_level_target_coords = batch.patch_level_target_coords
print(patch_level_reference_coords.shape, patch_level_target_coords.shape)

rotations = batch.rotations
print(rotations.shape)

In [ ]:
# batch = next(iter(dl))

count = 24 * 2

# radians to degrees
# print(batch.rotations[:count] * (180 / torch.pi))
# print(batch.rotations[:count])

show_batch(
    reference_patches, target_patches,
    patch_level_reference_coords, patch_level_target_coords,
    patch_level_target_coords,
    rotations_true=batch.rotations,
    rotations=None,
    limit_count=count, 
    n_columns=6,
)

In [28]:
# for index, batch in enumerate(dl):
#     print(f'-- Batch Index {index} -----')
#     reference_patches = batch.reference_patches
#     target_patches = batch.target_patches
#     print(reference_patches.shape, target_patches.shape) 

#     patch_level_reference_coords = batch.patch_level_reference_coords
#     patch_level_target_coords = batch.patch_level_target_coords
#     print(patch_level_reference_coords.shape, patch_level_target_coords.shape)

#     rotations = batch.rotations
#     print(rotations.shape)

In [29]:
dm.teardown()

# Check Model

In [30]:
from src import Light
light = Light()

In [ ]:
from utils import count_params
count_params(light.model)

In [ ]:
print('Model Inputs')
print(f'reference_patches shape : {reference_patches.shape}')
print(f'target_patches shape : {target_patches.shape}')
print(f'patch_level_reference_coords shape : {patch_level_reference_coords.shape}')

reference_patches, target_patches, patch_level_reference_coords = reference_patches.to('cuda'), target_patches.to('cuda'), patch_level_reference_coords.to('cuda')

print('Model Outputs')

target_coords, target_rotation, target_confidence = light(
    reference_patches, 
    target_patches, 
    patch_level_reference_coords
)

print(f'target_coords shape : {target_coords.shape}')
print(f'target_rotation shape : {target_rotation.shape}')
print(f'target_confidence shape : {target_confidence.shape}')

# Confidence

In [ ]:
patch_level_reference_coords[:1], target_coords[:1]

In [34]:
std_dev = 6.0
covariance_matrix = torch.diag(torch.tensor([std_dev ** 2, std_dev ** 2], device=patch_level_reference_coords.device))

gaussian = torch.distributions.MultivariateNormal(
    loc=patch_level_reference_coords, 
    covariance_matrix=covariance_matrix
)

In [35]:
# X = (target_coords + 1) * 15.5

# X = patch_level_reference_coords + 10  # for testing std dev

d = torch.arange(patch_level_reference_coords.shape[0], device=patch_level_reference_coords.device).unsqueeze(1)
X = patch_level_reference_coords + d  # for testing std dev

raw_probabilities = torch.exp(gaussian.log_prob(X))

In [36]:
# Y = patch_level_reference_coords / 31.0
# max_probabilities = torch.exp(gaussian.log_prob(Y))
max_probabilities = torch.exp(gaussian.log_prob(patch_level_reference_coords))

# Normalize probabilities to make p = 1 when coords_pred = coords
normalized_probabilities = raw_probabilities / (max_probabilities + 0.000001)

In [ ]:
normalized_probabilities[:15]